# Lesson 02 - Дослідження Microsoft Agent Framework

**Microsoft Agent Framework (MAF)** — це уніфікована структура для створення AI-агентів. Вона забезпечує чисту, композиційну архітектуру з чотирма основними блоками:

- **Клієнт** – підключається до кінцевої точки AI-моделі та обробляє комунікацію
- **Агент** – обгортає клієнта з інструкціями та визначеннями інструментів
- **Інструменти** – розширюють можливості агента за допомогою користувацьких функцій, які модель може викликати
- **Сесія** – зберігає історію розмови для багатоетапних взаємодій

У цьому уроці ми створимо **агента для бронювання подорожей**, який перевірятиме доступність напрямків, використовуючи ці концепції.


## Налаштування


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## Розуміння архітектури Agent Framework

Microsoft Agent Framework слідує багатошаровій архітектурі:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **Клієнт** – `AzureAIProjectAgentProvider` підключається до розгортання Azure OpenAI. Він обробляє автентифікацію, форматування запитів та розбір відповідей.
2. **Агент** – Створюється з клієнта через `provider.create_agent()`, агент поєднує доступ до моделі з інструкціями (системною підказкою) та інструментами.
3. **Інструменти** – Python-функції, декоровані `@tool`, які агент може викликати для виконання дій або отримання даних.
4. **Сесія** – Об’єкт `AgentSession` (створений через `agent.create_session()`), який зберігає історію діалогу, що дозволяє ведення багатоходового діалогу, де агент пам’ятає попередній контекст.

Давайте побудуємо кожен рівень крок за кроком.


In [ ]:
# Create the client – this is the connection to the AI model
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Додавання інструментів за допомогою декоратора @tool

Інструменти дозволяють агентам виконувати дії, окрім генерації тексту. Декоратор `@tool` перетворює звичайну функцію Python у те, що агент може викликати.

Ключові моменти:
- Використовуйте `Annotated[type, "description"]`, щоб модель розуміла кожен параметр.
- Докстрінг стає описом інструменту, який бачить модель.
- `approval_mode="never_require"` означає, що інструмент запускається автоматично без підтвердження користувача.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## Створення агента з інструментами

Тепер ми об’єднуємо клієнта, інструкції та інструменти в агента. `instructions` виступають як системний підказчик — вони визначають персоналію та поведінку агента.


In [ ]:
agent = await provider.create_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## Багатокрокові розмови з сесіями

`AgentSession` (створена за допомогою `agent.create_session()`) відстежує всі повідомлення у розмові. Передаючи ту саму сесію в кожен виклик `agent.run()`, агент має доступ до повної історії розмови і може посилатися на попередні повідомлення.

Ми передаємо `tools=[check_destination_availability]`, щоб агент міг викликати наш перевірник доступності під час кожного кроку.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## Підсумок

У цьому уроці ви ознайомилися з чотирма основами Microsoft Agent Framework:

| Поняття | Чому ви навчилися |
|---------|------------------|
| **Клієнт** | `AzureAIProjectAgentProvider` підключається до Azure OpenAI за допомогою аутентифікації на основі облікових даних |
| **Агент** | `provider.create_agent()` об’єднує підключення до моделі з інструкціями та ім’ям |
| **Інструменти** | Декоратор `@tool` надає Python-функції для виклику агентом |
| **Сесія** | `agent.create_session()` зберігає історію розмови протягом кількох кроків |

Ці будівельні блоки поєднуються для створення агентів, які можуть вести природні розмови, викликати зовнішні функції та зберігати контекст — основу для більш складних агентних схем у наступних уроках.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Відмова від відповідальності**:
Цей документ був перекладений за допомогою сервісу автоматичного перекладу [Co-op Translator](https://github.com/Azure/co-op-translator). Хоча ми прагнемо до точності, будь ласка, майте на увазі, що автоматичні переклади можуть містити помилки або неточності. Оригінальний документ рідною мовою слід вважати авторитетним джерелом. Для критично важливої інформації рекомендується користуватися послугами професійного людського перекладу. Ми не несемо відповідальності за будь-які непорозуміння або помилкові тлумачення, що виникли внаслідок використання цього перекладу.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
